In [22]:
from langgraph.graph import StateGraph,START,END
from typing import TypedDict,Annotated
from langchain_core.messages import BaseMessage,HumanMessage
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver

In [23]:
load_dotenv
model = ChatOpenAI()

In [24]:
class ChatState(TypedDict):
    messages:Annotated[list[BaseMessage],add_messages]
    

In [25]:
def chat_node(state:ChatState):
    messages = state['messages']
    
    response = model.invoke(messages)
    
    return {
        'messages':[response]
    }

In [26]:
check_pointer = MemorySaver()

graph =StateGraph(ChatState)

graph.add_node('chat_node',chat_node)

graph.add_edge(START,'chat_node')
graph.add_edge('chat_node',END)


chatbot = graph.compile(checkpointer=check_pointer)

In [27]:
# initial_state={
#     'messages':[HumanMessage('What is capital of haryana')]
# }
# result = chatbot.invoke(initial_state)


In [28]:
thread_id = '1'
while True:
    user_message =input('Type here:')
    print('User: ',user_message)
    if user_message.strip().lower() in ['exit','quit','bye']:
        break
    config ={
        'configurable':{
            'thread_id':thread_id
        }
    }
    response = chatbot.invoke({'messages':[HumanMessage(content=user_message)]},config=config)
    
    print('AI:', response['messages'][-1].content)

User:  Hi, my name is abhishek
AI: Hello, Abhishek! How can I assist you today?
User:  what is my name
AI: Your name is Abhishek.
User:  can you add 10 to 1000
AI: Sure! 1000 + 10 = 1010.
User:  can you multiply 100 with prev result
AI: Of course! 1010 * 100 = 101000.
User:  bye
